In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
import time, psutil
import optuna
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.integration.keras import TuneReportCallback
from ray.tune import TuneConfig
from ray.tune import Tuner, TuneConfig
from ray.air.config import RunConfig
from ray.tune.progress_reporter import CLIReporter

In [ ]:

# Parametry
N = 10000
rng = np.random.default_rng(42)

# Klasy: 0 - karpiowy, 1 - liniowy, 2 - pstrągowy
classes = rng.choice([0, 1, 2], size=N, p=[0.4, 0.35, 0.25])

# Cechy środowiskowe i biologiczne - bardziej nakładające się
temperatura = np.select([classes==0, classes==1, classes==2],
                        [rng.normal(20, 3, N), rng.normal(18, 3, N), rng.normal(15, 3, N)])
ph = np.select([classes==0, classes==1, classes==2],
               [rng.normal(7.3, 0.3, N), rng.normal(7.1, 0.3, N), rng.normal(6.9, 0.3, N)])
tlen = np.select([classes==0, classes==1, classes==2],
                 [rng.normal(6.8, 0.6, N), rng.normal(7.2, 0.6, N), rng.normal(7.8, 0.6, N)])
azotany = np.select([classes==0, classes==1, classes==2],
                    [rng.normal(2.8, 0.9, N), rng.normal(2.6, 0.9, N), rng.normal(2.0, 0.9, N)])
fosforany = np.select([classes==0, classes==1, classes==2],
                      [rng.normal(0.55, 0.2, N), rng.normal(0.5, 0.2, N), rng.normal(0.45, 0.2, N)])
przezroczystosc = np.select([classes==0, classes==1, classes==2],
                            [rng.normal(1.5, 0.4, N), rng.normal(1.6, 0.4, N), rng.normal(1.8, 0.4, N)])
gleba = np.select([classes==0, classes==1, classes==2],
                  [rng.normal(5.0, 1.2, N), rng.normal(4.8, 1.2, N), rng.normal(5.3, 1.2, N)])
powierzchnia = np.select([classes==0, classes==1, classes==2],
                         [rng.normal(2.2, 0.5, N), rng.normal(2.0, 0.5, N), rng.normal(2.4, 0.5, N)])
glebokosc = np.select([classes==0, classes==1, classes==2],
                      [rng.normal(1.8, 0.4, N), rng.normal(2.0, 0.4, N), rng.normal(2.3, 0.4, N)])
roslinnosc = np.select([classes==0, classes==1, classes==2],
                       [rng.normal(0.7, 0.2, N), rng.normal(0.8, 0.2, N), rng.normal(0.6, 0.2, N)])
przeplyw = np.select([classes==0, classes==1, classes==2],
                     [rng.normal(0.35, 0.1, N), rng.normal(0.4, 0.1, N), rng.normal(0.5, 0.1, N)])
srednia_waga_ryb = np.select([classes==0, classes==1, classes==2],
                             [rng.normal(1.0, 0.3, N), rng.normal(0.8, 0.3, N), rng.normal(0.6, 0.3, N)])

# DataFrame
df = pd.DataFrame({
    "temperatura_wody": temperatura,
    "pH": ph,
    "tlen": tlen,
    "azotany": azotany,
    "fosforany": fosforany,
    "przezroczystość": przezroczystosc,
    "gleba": gleba,
    "powierzchnia_stawu": powierzchnia,
    "głębokość": glebokosc,
    "roślinność": roslinnosc,
    "przepływ": przeplyw,
    "średnia_waga_ryb": srednia_waga_ryb,
    "typ_stawu": classes
})

df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample_df = df.sample(5000, random_state=42)

palette_map = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}

plt.figure(figsize=(8, 6))
ax = sns.scatterplot(
    data=sample_df,
    x="temperatura_wody",
    y="tlen",
    hue="typ_stawu",
    palette=palette_map,
    alpha=0.7
)

# poprawia opisy w legendzie bez psucia kolorów
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["karpiowy (0)", "liniowy (1)", "pstrągowy (2)"], title="Typ stawu")

plt.title("Rozkład danych dla przykładowych cech")
plt.xlabel("Temperatura wody [°C]")
plt.ylabel("Zawartość tlenu [mg/L]")
plt.show()


In [ ]:
#zaokrąglenie dla czytelności
df = df.round(3)
df.head()

In [ ]:
print(df.info())


In [ ]:
print(df.describe().T)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Podział 70/30: trening/test
X = df.drop(columns=["typ_stawu"])
y = df["typ_stawu"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# walidacja wydzielona z treningu  20% z treningu)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print("Treningowe:", X_train.shape)
print("Walidacyjne:", X_val.shape)
print("Testowe:", X_test.shape)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_val_scaled   = scaler.transform(X_val)
x_test_scaled  = scaler.transform(X_test)


## Bazowy model MLP

In [ ]:
start = time.perf_counter()
process = psutil.Process()

model = keras.Sequential([
    layers.Input(shape=(x_train_scaled.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    x_train_scaled, y_train,
    validation_data=(x_val_scaled, y_val), 
    epochs=30,
    batch_size=64,
    verbose=1,
    callbacks=[callback]
)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

print(f"Czas uczenia: {elapsed:.2f}s")
print(f"Zużycie pamięci: {mem_used:.2f} MB")

# Ocena na zbiorze testowym (tylko na końcu)
test_loss, test_acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print(f"Wynik testowy - loss: {test_loss:.4f}, accuracy: {test_acc:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# predykcje prawdopodobieństw klas
y_prob = model.predict(x_test_scaled, verbose=0)

# loss dla każdej próbki (bez uśredniania)
sce = tf.keras.losses.SparseCategoricalCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
loss_per_sample = sce(y_test, y_prob).numpy()

plt.figure(figsize=(8, 5))
plt.hist(loss_per_sample, bins=40, edgecolor='black')
plt.xlabel("Strata (cross-entropy) na próbkę")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()


## Random Search

In [ ]:
def build_model(hp):
    model = keras.Sequential([
        layers.Input(shape=(x_train_scaled.shape[1],)),

        # Pierwsza warstwa
        layers.Dense(
            units=hp.Int('units_1', min_value=32, max_value=256, step=32),
            activation='relu'
        ),

        # Dropout
        layers.Dropout(
            rate=hp.Float('dropout', min_value=0.1, max_value=0.5, step=0.1)
        ),

        # Druga warstwa
        layers.Dense(
            units=hp.Int('units_2', min_value=16, max_value=256, step=16),
            activation='relu'
        ),

        layers.Dense(3, activation='softmax')
    ])

    # batch_size jako hiperparametr
    hp.Choice('batch_size', values=[32, 64, 128, 256])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Float('lr', 1e-5, 1e-2, sampling='log')
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory="C:\\Users\\ASUS ZENBOOK\\Desktop\\tuning_results",
    project_name='random_search71'
)

callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

start = time.perf_counter()
process = psutil.Process()

tuner.search(
    x_train_scaled, y_train,
    validation_data=(x_val_scaled, y_val),   
    epochs=30,
    callbacks=[callback],
    verbose=1
)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

best_hp = tuner.get_best_hyperparameters(1)[0]
best_batch = best_hp.get('batch_size')
print(f"\nWybrany batch_size: {best_batch}")

best_model = tuner.hypermodel.build(best_hp)
history = best_model.fit(
    x_train_scaled, y_train,
    validation_data=(x_val_scaled, y_val),
    epochs=30,
    batch_size=best_batch,
    callbacks=[callback],
    verbose=0
)


# OCENA NA ZBIORZE TESTOWYM (tylko na końcu)
test_loss, test_acc = best_model.evaluate(x_test_scaled, y_test, verbose=0)

print("\n=== RANDOM SEARCH (MLP) ===")
print("Najlepsze hiperparametry:")
for param, value in best_hp.values.items():
    print(f"  {param}: {value}")

print(f"Wynik walidacyjny (val_accuracy z tunera): {best_hp.get('val_accuracy') if 'val_accuracy' in best_hp.values else 'sprawdź w logach'}")
print(f"Wynik testowy accuracy: {test_acc:.4f}")
print(f"Czas przeszukiwania: {elapsed:.2f} s | Zużycie pamięci: {mem_used:.2f} MB")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import tensorflow as tf

y_prob = best_model.predict(x_test_scaled, verbose=0)

sce = tf.keras.losses.SparseCategoricalCrossentropy(
    reduction=tf.keras.losses.Reduction.NONE
)
loss_per_sample = sce(y_test, y_prob).numpy()

plt.figure(figsize=(8, 5))
plt.hist(loss_per_sample, bins=40, edgecolor='black')
plt.xlabel("Strata na próbkę (cross-entropy)")
plt.ylabel("Liczba próbek")

plt.grid(True, alpha=0.3)
plt.show()


## Hyperband 

In [ ]:
def build_model(hp):
    model = keras.Sequential([
        layers.Input(shape=(x_train_scaled.shape[1],)),

        # Pierwsza warstwa ukryta
        layers.Dense(
            units=hp.Int('units_1', min_value=32, max_value=256, step=32),
            activation='relu'
        ),

        # Dropout
        layers.Dropout(
            rate=hp.Float('dropout', min_value=0.1, max_value=0.5, step=0.1)
        ),

        # Druga warstwa ukryta
        layers.Dense(
            units=hp.Int('units_2', min_value=16, max_value=128, step=16),
            activation='relu'
        ),

        layers.Dense(3, activation='softmax')
    ])

    hp.Choice('batch_size', values=[32, 64, 128, 256])

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=hp.Float('lr', 1e-5, 1e-2, sampling='log')
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=30,
    factor=3,
    directory="C:\\Users\\ASUS ZENBOOK\\Desktop\\tuning_results",
    project_name='hyperband4'
)

callback = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

start = time.perf_counter()
process = psutil.Process()

tuner.search(
    x_train_scaled, y_train,
    validation_data=(x_val_scaled, y_val),   
    epochs=30,
    callbacks=[callback],
    verbose=1
)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

best_hp = tuner.get_best_hyperparameters(1)[0]
best_batch = best_hp.get('batch_size')
print(f"\nWybrany batch_size: {best_batch}")

best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(
    x_train_scaled, y_train,
    validation_data=(x_val_scaled, y_val), 
    epochs=30,
    batch_size=best_batch,
    callbacks=[callback],
    verbose=0
)

# OCENA NA ZBIORZE TESTOWYM (tylko na końcu)
test_loss, test_acc = best_model.evaluate(x_test_scaled, y_test, verbose=0)

print("\n=== HYPERBAND (MLP) ===")
print("Najlepsze hiperparametry:")
for param, value in best_hp.values.items():
    print(f"  {param}: {value}")

print(f"Wynik testowy accuracy: {test_acc:.4f}")
print(f"Czas przeszukiwania: {elapsed:.2f} s | Zużycie pamięci: {mem_used:.2f} MB")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

import tensorflow as tf

y_prob = best_model.predict(x_test_scaled, verbose=0)

sce = tf.keras.losses.SparseCategoricalCrossentropy(
    reduction=tf.keras.losses.Reduction.NONE
)
loss_per_sample = sce(y_test, y_prob).numpy()

plt.figure(figsize=(8, 5))
plt.hist(loss_per_sample, bins=40, edgecolor='black')
plt.xlabel("Strata na próbkę (cross-entropy)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()


## Optuna


In [ ]:
def objective(trial):
    n_units_1 = trial.suggest_int('units_1', 32, 256, step=32)
    n_units_2 = trial.suggest_int('units_2', 16, 256, step=16)
    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])

    model = keras.Sequential([
        layers.Input(shape=(x_train_scaled.shape[1],)),
        layers.Dense(n_units_1, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(n_units_2, activation='relu'),
        layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    callback = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        x_train_scaled, y_train,
        validation_data=(x_val_scaled, y_val), 
        epochs=30,
        batch_size=batch_size,
        callbacks=[callback],
        verbose=0
    )

    return max(history.history['val_accuracy'])


start = time.perf_counter()
process = psutil.Process()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

print("\n=== OPTUNA (MLP) ===")
print("Najlepsze hiperparametry:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

print(f"Najlepszy wynik walidacyjny: {study.best_value:.4f}")
print(f"Czas przeszukiwania: {elapsed:.2f} s | Zużycie pamięci: {mem_used:.2f} MB")

# trening finalnego modelu na najlepszych HP i test na końcu:
best_params = study.best_params

best_model = keras.Sequential([
    layers.Input(shape=(x_train_scaled.shape[1],)),
    layers.Dense(best_params['units_1'], activation='relu'),
    layers.Dropout(best_params['dropout']),
    layers.Dense(best_params['units_2'], activation='relu'),
    layers.Dense(3, activation='softmax')
])

best_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_params['lr']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callback_final = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = best_model.fit(
    x_train_scaled, y_train,
    validation_data=(x_val_scaled, y_val),
    epochs=30,
    batch_size=best_params['batch_size'],
    callbacks=[callback_final],
    verbose=0
)

test_loss, test_acc = best_model.evaluate(x_test_scaled, y_test, verbose=0)
print(f"Wynik testowy accuracy: {test_acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

import tensorflow as tf

y_prob = best_model.predict(x_test_scaled, verbose=0)

sce = tf.keras.losses.SparseCategoricalCrossentropy(
    reduction=tf.keras.losses.Reduction.NONE
)
loss_per_sample = sce(y_test, y_prob).numpy()

plt.figure(figsize=(8, 5))
plt.hist(loss_per_sample, bins=40, edgecolor='black')
plt.xlabel("Strata na próbkę (cross-entropy)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()


## Ray Tune

In [ ]:
from ray.air import session

In [ ]:
def build_model(config):
    model = keras.Sequential([
        layers.Input(shape=(x_train_scaled.shape[1],)),
        layers.Dense(config["units_1"], activation="relu"),
        layers.Dropout(config["dropout"]),
        layers.Dense(config["units_2"], activation="relu"),
        layers.Dense(3, activation="softmax")
    ])
    opt = keras.optimizers.Adam(learning_rate=config["lr"])
    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


start = time.perf_counter()
process = psutil.Process()


def train_model(config):
    model = build_model(config)

    es = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        x_train_scaled, y_train,
        validation_data=(x_val_scaled, y_val),  
        epochs=30,
        batch_size=config["batch_size"],
        verbose=0,
        callbacks=[es]
    )

    val_acc = float(np.nanmax(history.history["val_accuracy"]))
    session.report({"val_accuracy": val_acc})


param_space = {
    "units_1": tune.choice([32, 64, 128]),
    "units_2": tune.choice([16, 32, 64]),
    "dropout": tune.uniform(0.1, 0.5),
    "lr": tune.loguniform(1e-5, 1e-2),
    "batch_size": tune.choice([32, 64, 128, 256]),
}

tuner = Tuner(
    tune.with_parameters(train_model),
    param_space=param_space,
    tune_config=TuneConfig(
        metric="val_accuracy",
        mode="max",
        num_samples=20,
    ),
)

results = tuner.fit()

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

best_result = results.get_best_result(metric="val_accuracy", mode="max")

print("\n=== RAY TUNE (MLP) ===")
print("Najlepsze hiperparametry:", best_result.config)
print("Najlepszy wynik walidacyjny:", best_result.metrics["val_accuracy"])
print(f"\nCzas uczenia: {elapsed:.2f}s")
print(f"Zużycie pamięci: {mem_used:.2f} MB")

#trening finalny na najlepszych HP i test na końcu:
best_config = best_result.config
best_model = build_model(best_config)

es_final = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = best_model.fit(
    x_train_scaled, y_train,
    validation_data=(x_val_scaled, y_val),
    epochs=30,
    batch_size=best_config["batch_size"],
    verbose=0,
    callbacks=[es_final]
)

test_loss, test_acc = best_model.evaluate(x_test_scaled, y_test, verbose=0)
print(f"Wynik testowy accuracy: {test_acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

import tensorflow as tf

y_prob = best_model.predict(x_test_scaled, verbose=0)

sce = tf.keras.losses.SparseCategoricalCrossentropy(
    reduction=tf.keras.losses.Reduction.NONE
)
loss_per_sample = sce(y_test, y_prob).numpy()

plt.figure(figsize=(8, 5))
plt.hist(loss_per_sample, bins=40, edgecolor='black')
plt.xlabel("Strata na próbkę (cross-entropy)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()
